<a href="https://colab.research.google.com/github/oste3224/oste3224.github.io/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install geoalchemy2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 2.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from geoalchemy2 import Geometry, WKTElement
from shapely.geometry import Point, Polygon, MultiPolygon

ModuleNotFoundError: No module named 'geoalchemy2'

## Task 1. Import & Clean datasets

### Step 1: load and clean `Business.csv`

In [ ]:
# Read CSV with proper dtypes for codes/names
business_orignial = pd.read_csv('Businesses.csv', dtype={
    'industry_code': str,
    'industry_name': str,
    'sa2_code': str,
    'sa2_name': str
})

# Avoid modifying the original dataframe
business = business_orignial.copy()

# Check missing values
print("=== Missing values per column ===")
print(business.isnull().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'Businesses.csv'

In [ ]:
# Ensure the numeric columns are integers
numeric_cols = [
    '0_to_50k_businesses',
    '50k_to_200k_businesses',
    '200k_to_2m_businesses',
    '2m_to_5m_businesses',
    '5m_to_10m_businesses',
    '10m_or_more_businesses',
    'total_businesses'
]
business[numeric_cols] = business[numeric_cols].astype(int)

# Check for duplicates
print("=== Duplicates ===")
print(business.duplicated().sum())
print("=== Duplicates industry_code and sa2_code ===")
print(business.duplicated(subset=['industry_code', 'sa2_code']).sum())

In [ ]:
# Verify that sum of segments equals total_businesses
segment_cols = numeric_cols[:-1]
business['computed_total'] = business[segment_cols].sum(axis=1)
mismatch = business[business['computed_total'] != business['total_businesses']]
print(f"Rows with mismatched total_businesses: {len(mismatch)}")

In [ ]:
print(mismatch[['industry_code','sa2_code','computed_total','total_businesses']])

# Replace mismatched total_businesses with computed totals
print("\nReplacing mismatched total_businesses with computed totals...")
business['total_businesses'] = business['computed_total']

# Verify all mismatches are fixed
mismatch_after = business[business['computed_total'] != business['total_businesses']]
print(f"Rows with mismatched total_businesses after correction: {len(mismatch_after)}")

# Final: drop the helper column
business = business.drop(columns=['computed_total'])

In [ ]:
# View data types
print("=== Data types ===")
print(business.dtypes)

# Read the dataframe
business.head()

### Step 2: load and clean `SA2_region` dataset

In [ ]:
# Load the SA2 shapefile data
sa2_shapefile_path = 'SA2_2021_AUST_GDA2020.shp'
sa2_gdf_original = gpd.read_file(sa2_shapefile_path)

In [ ]:
# Avoid modifying the original dataframe
sa2_gdf = sa2_gdf_original.copy()

# Display basic information about the shapefile
print("=== SA2 Shapefile Information ===")
print(f"Number of SA2 regions: {len(sa2_gdf)}")
print(f"Coordinate Reference System (CRS): {sa2_gdf.crs}")
print("\nColumns in the shapefile:")
print(sa2_gdf.columns.tolist())

# Filter to include only Greater Sydney
sydney_sa2_gdf = sa2_gdf[sa2_gdf['GCC_NAME21'] == 'Greater Sydney']

# Update the main dataframe to only include Sydney
sa2_gdf = sydney_sa2_gdf

NameError: name 'sa2_gdf_original' is not defined

In [ ]:
# Remove the columns which are not relevant to the analysis
coulmns_to_remove = ['CHG_FLAG21', 'CHG_LBL21', 'SA3_CODE21', 'SA3_NAME21', 'STE_CODE21', 'STE_NAME21', 'AUS_CODE21', 'AUS_NAME21', 'LOCI_URI21']
sa2_gdf.drop(columns=coulmns_to_remove, inplace=True)

# Reset the index
sa2_gdf = sa2_gdf.reset_index(drop=True)

sa2_gdf.head()

In [ ]:
# Check for any issues that may need cleaning
print("=== SA2 Shapefile Data Quality Checks ===")
print("Missing values:")
print(sa2_gdf.isnull().sum())

# Check for invalid geometries
print("\nInvalid geometries:")
invalid_geoms = sa2_gdf[~sa2_gdf.geometry.is_valid]
print(f"Number of invalid geometries: {len(invalid_geoms)}")
if len(invalid_geoms) > 0:
    print(invalid_geoms[['SA2_CODE21', 'SA2_NAME21']])

# Check for duplicate SA2 codes
print("\nDuplicate SA2 codes:")
duplicate_sa2_codes = sa2_gdf[sa2_gdf.duplicated(subset=['SA2_CODE21'], keep=False)]
print(f"Number of duplicate SA2 codes: {len(duplicate_sa2_codes)}")
if len(duplicate_sa2_codes) > 0:
    print(duplicate_sa2_codes[['SA2_CODE21', 'SA2_NAME21']])

In [ ]:
# Get a list of all columns except geometry
non_geom_columns = [col for col in sa2_gdf.columns if col != 'geometry']

# Ensure all non-geometry columns of type are string, except 'AREASQKM21'
for col in non_geom_columns:
    if col != 'AREASQKM21':
        sa2_gdf[col] = sa2_gdf[col].astype(str)

# Create a dictionary to rename columns
rename_dict = {}
for col in sa2_gdf.columns:
    if col != 'geometry':
        # Remove '21' suffix if present and convert to lowercase
        new_name = col.lower().replace('21', '') if col.endswith('21') else col.lower()
        rename_dict[col] = new_name

# Rename the columns
sa2_gdf = sa2_gdf.rename(columns=rename_dict)

print(f"Number of SA2 regions: {len(sa2_gdf)}")
print("Columns in the cleaned shapefile:")
print(sa2_gdf.columns.tolist())

NameError: name 'sa2_gdf' is not defined

In [ ]:
# Visualize the SA2 regions with light fill and clear boundaries
plt.figure(figsize=(12, 10))
sa2_gdf.plot(
    color='lightgray',  # Single light color for all regions
    edgecolor='black',
    linewidth=0.5
)
plt.title('SA2 Regions in Greater Sydney (2021)', fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# This function is used to wrap the geometry column with WKTElement for the polygon data
def create_wkt_element(geom, srid):
    if geom.geom_type == 'Polygon':
        geom = MultiPolygon([geom])
    return WKTElement(geom.wkt, srid)

### Step 3: load and clean `Stops.csv`

In [ ]:
# Define the file path for the new dataset
stops_file_path = 'Stops.txt'

# Load the dataset
stops_df_original = pd.read_csv(stops_file_path, sep = ',', quotechar= '"')

# Check missing values
print("=== Missing values per column ===")
print(stops_df_original.isnull().sum())

# Display the first few rows
stops_df_original.head()

In [ ]:
# Avoid modifying the original dataframe
stops_df = stops_df_original.copy()

# Handle Missing Values
critical_cols = ['stop_id', 'stop_name', 'stop_lat', 'stop_lon']
initial_rows = len(stops_df)
stops_df.dropna(subset=critical_cols, inplace=True)
print("=== Missing value rows dropped ===")
print(f"Dropped {initial_rows - len(stops_df)} rows")


In [ ]:
# Drop columns which are not relevant to the analysis
cols_to__drop = ['stop_code', 'location_type', 'parent_station', 'platform_code', 'wheelchair_boarding']
stops_df.drop(columns=cols_to__drop, inplace=True)
stops_df.head()

In [ ]:
# Check/Correct Data Types (Example)
print("=== Data types ===")
print(stops_df.dtypes)

In [ ]:
dup_rows = stops_df[stops_df.duplicated(subset=['stop_id'], keep=False)]

# How many total rows are in these duplicate groups?
print("=== Duplicate stop_id ===")
print(f"Found {len(dup_rows)} rows belonging to duplicate stop_id values.")

NameError: name 'stops_df' is not defined

In [ ]:
# Combine the lat and lon as geometry
stops_df['geometry'] = gpd.points_from_xy(stops_df.stop_lon, stops_df.stop_lat)
stops_df = stops_df.drop(columns=['stop_lon', 'stop_lat'])

srid = 7844
stops_df['geometry'] = stops_df['geometry'].apply(lambda x: x.wkt)
stops_df.head()

### Step 4: load and clean `Schools` datasets

In [ ]:
# Load the Schools shapefile data
future_school_shapefile_path = 'catchments/catchments_future.shp'
future_gdf_original = gpd.read_file(future_school_shapefile_path)

primary_school_shapefile_path = 'catchments/catchments_primary.shp'
primary_gdf_original = gpd.read_file(primary_school_shapefile_path)

secondary_school_shapefile_path = 'catchments/catchments_secondary.shp'
secondary_gdf_original = gpd.read_file(secondary_school_shapefile_path)

In [ ]:
# Avoid modifying the original dataframe
future_gdf = future_gdf_original.copy()
primary_gdf = primary_gdf_original.copy()
secondary_gdf = secondary_gdf_original.copy()

# Remove all columns are not relevant to the analysis
cols_to_remove = ['ADD_DATE','KINDERGART', 'YEAR1','YEAR2', 'YEAR3', 'YEAR4', 'YEAR5', 'YEAR6', 'YEAR7', 'YEAR8', 'YEAR9', 'YEAR10', 'YEAR11', 'YEAR12']
future_gdf.drop(columns=cols_to_remove, inplace=True)
primary_gdf.drop(columns=cols_to_remove + ['PRIORITY'], inplace=True)
secondary_gdf.drop(columns=cols_to_remove + ['PRIORITY'], inplace=True)

# Concatenate the three GeoDataFrames
schools_gdf = pd.concat([future_gdf, primary_gdf, secondary_gdf], ignore_index=True)

# Display the first few rows and info of the merged dataframe to verify
print("\n=== Head of schools_gdf ===")
schools_gdf

#### === Check if the rows with same id have same geometry ===

In [ ]:
# List unique CATCH_TYPE values
unique_catch_types = schools_gdf['CATCH_TYPE'].unique()
print("\n=== Unique CATCH_TYPE values in schools_gdf ===")
print(unique_catch_types)

# Remove rows where both USE_ID and geometry are identical duplicates
print(f"\nOriginal number of rows in schools_gdf: {len(schools_gdf)}")
duplicate_check_columns = ['USE_ID', 'geometry']
schools_gdf.drop_duplicates(subset=duplicate_check_columns, keep='first', inplace=True)

print(f"Number of rows after removing duplicates based on {duplicate_check_columns}: {len(schools_gdf)}")

# Remove rows that are completely duplicated across all columns
print(f"\nShape before removing duplicates: {schools_gdf.shape}")
schools_gdf.drop_duplicates(inplace=True)
print(f"Shape after removing duplicates: {schools_gdf.shape}")

In [ ]:
# Check the CRS of the schools_gdf
print(schools_gdf.crs)

In [ ]:
# Convert the CRS to EPSG: 7844
schools_gdf = schools_gdf.to_crs(epsg=7844)
print(schools_gdf.crs)

NameError: name 'schools_gdf' is not defined

In [ ]:
# Wrap the geometry column with WKTElement
schools_gdf['geom'] = schools_gdf['geometry'].apply(lambda x: create_wkt_element(geom=x,srid=7844))  # applying the function
schools_gdf = schools_gdf.drop(columns="geometry")
schools_gdf.head()

In [ ]:
schools_gdf = schools_gdf.rename(columns={
    'USE_ID': 'school_code',
    'CATCH_TYPE': 'catchment_type',
    'USE_DESC': 'school_name',
    'geom': 'geometry'
})

# Display the first few rows to confirm changes
print("=== schools_gdf after renaming columns: ===")
display(schools_gdf.head())


### Step 5: Load and clean `Population.csv`

In [ ]:
population_file_path = 'Population.csv'

# Load the dataset
population_df_original = pd.read_csv(population_file_path)
population_df = population_df_original.copy() # Work on a copy
print("=== Population.csv loaded successfully ===")
print(f"Shape of the dataframe: {population_df.shape}")

print("\n=== First 5 rows: ===")
display(population_df.head())


print("\n=== Data types: ===")
population_df.info() # .info() prints directly to stdout

print("\n=== Missing values per column: ===")
# Similarly, display() or print() for the Series of null counts
display(population_df.isnull().sum())

print("\n=== Duplicates ===")
print(population_df.duplicated().sum())


FileNotFoundError: [Errno 2] No such file or directory: 'Population.csv'

In [ ]:
# Verify that sum of segments equals total_people
segment_cols = population_df.columns[2:20]
population_df['computed_total'] = population_df[segment_cols].sum(axis=1)
mismatch = population_df[population_df['computed_total'] != population_df['total_people']]
print(f"Rows with mismatched total_people: {len(mismatch)}")

In [ ]:
# Keep only rows where 'total_people' is greater than or equal to 100
population_df = population_df.loc[population_df['total_people'] >= 100]
print(f"Rows after filtering: {len(population_df)}")

### Step 6: Load and clean `Income.csv`

In [ ]:
income_file_path = 'Income.csv'

income_df_original = pd.read_csv(income_file_path, dtype={'SA2_CODE_2021': str})
income_df = income_df_original.copy() # Work on a copy

print("=== Income.csv loaded successfully ===")
print(f"Shape of the dataframe: {income_df.shape}")
print("\n=== First 5 rows: ===")
display(income_df.head())


In [ ]:
print("\n=== Data types: ===")
income_df.info()

print("\n=== Missing values per column: ===")
display(income_df.isnull().sum())

print("\n=== Duplicates before any cleaning: ===")
print(f"Number of duplicate rows: {income_df.duplicated().sum()}")


In [ ]:
rename_map = {'sa2_code21': 'sa2_code'}

income_df.rename(columns=rename_map, inplace=True)
print("\n=== Column names after initial rename attempt: ===")
print(list(income_df.columns))

## Task 2: NSW Points of Interest API

**Task 2.i:** Develop a function that returns all points of interests from the API within a specified bounding box.

In [ ]:
import requests
import json
import time

def get_poi_by_bbox(bbox, filters=None):
    """
    Fetches Points of Interest (POIs) from the NSW POI API within a specified bounding box.

    Args:
        bbox (dict): A dictionary defining the bounding box with keys
                     'xmin', 'ymin', 'xmax', 'ymax'.
        filters (dict, optional): Additional filters for the API query. Defaults to None.

    Returns:
        list: A list of POI features if the request is successful, otherwise None.
              Each feature is a dictionary.
    """
    base_url = 'https://maps.six.nsw.gov.au/arcgis/rest/services/public/NSW_POI/MapServer/0/query'

    # The API expects the geometry parameter as a string like:
    # "xmin:150.0,ymin:-34.0,xmax:151.0,ymax:-33.0"
    # Ensure bbox values are rounded or formatted as needed if they come from GeoDataFrame bounds
    geometry_param = f"\"xmin\":{bbox['xmin']},\"ymin\":{bbox['ymin']},\"xmax\":{bbox['xmax']},\"ymax\":{bbox['ymax']}"

    params = {
        'geometry': geometry_param,
        'geometryType': 'esriGeometryEnvelope', # Specify that the geometry is a bounding box
        'inSR': '7844',
        'outFields': '*', # Request all fields
        'returnGeometry': 'true', # Request geometry for the POIs
        'outSR': '7844', # Request POI geometry in GDA94
        'f': 'json' # Request format
    }

    if filters:
        params.update(filters)

    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status() # Raises an HTTPError for bad responses (4XX or 5XX)
        data = response.json()
        if 'features' in data:
            return data['features']
        else:
            print(f"Warning: 'features' not found in API response for bbox {bbox}. Response: {data}")
            return [] # Return empty list if features are missing but no error raised
    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        print(f"Request URL: {response.url}") # Print the failing URL for debugging
        print(f"Response content: {response.text}")
        return None
    except requests.exceptions.RequestException as req_err:
        print(f"Request error occurred: {req_err}")
        return None
    except json.JSONDecodeError:
        print("Error decoding JSON from API response.")
        print(f"Response content: {response.text}")
        return None

In [ ]:
# Find the sa4_code for Northern Beaches
try:
    northern_beaches_code = sa2_gdf[sa2_gdf['sa4_name'] == 'Sydney - Northern Beaches']['sa4_code'].unique()
    if len(northern_beaches_code) > 0:
        nb_code = northern_beaches_code[0]
        print(f"The sa4_code for 'Northern Beaches' is: {nb_code}")
    else:
        print("Could not find sa4_code for 'Northern Beaches'. Available SA4 names:")
        print(sa2_gdf['sa4_name'].unique())
        nb_code = None # Set to None if not found
except NameError:
    print("Error: sa2_gdf is not defined. Make sure the preceding cells are executed.")
    nb_code = None
except KeyError:
    print("Error: 'sa4_name' or 'sa4_code' column not found in sa2_gdf.")
    nb_code = None

The sa4_code for 'Northern Beaches' is: 122


In [ ]:
# Task 2.ii: Loop through SA2 regions in the selected SA4, get POIs using bounding box

# Define the selected SA4 code (found in the previous cell)
selected_sa4_code = '122' # For Sydney - Northern Beaches
selected_sa4_name = 'Sydney - Northern Beaches'

# Filter sa2_gdf for the selected SA4 region
# Make sure sa2_gdf is the version after cell [40] (Greater Sydney, cleaned columns, shapely geometry)
sa2_in_selected_sa4 = sa2_gdf[sa2_gdf['sa4_code'] == selected_sa4_code].copy()

print(f"Processing {len(sa2_in_selected_sa4)} SA2 regions in SA4: {selected_sa4_name} ({selected_sa4_code})")

# --- Calculate bounding boxes ---
# The .bounds property gives (minx, miny, maxx, maxy)
sa2_in_selected_sa4['bbox_coords'] = sa2_in_selected_sa4.geometry.apply(lambda geom: geom.bounds)

# --- Loop through SA2s and fetch POIs ---
all_pois_in_selected_sa4 = [] # List to store all POIs found

for index, row in sa2_in_selected_sa4.iterrows():
    sa2_code = row['sa2_code']
    sa2_name = row['sa2_name']
    bounds = row['bbox_coords']
    sa2_geometry = row['geometry']

    # Format the bounding box for the API function
    bbox_dict = {'xmin': bounds[0], 'ymin': bounds[1], 'xmax': bounds[2], 'ymax': bounds[3]}

    print(f"\nFetching POIs for SA2: {sa2_name} ({sa2_code}) using bbox...")

    # Wait for 1 second before the API call
    time.sleep(1)

    # Ensure the get_poi_by_bbox function is defined and works correctly
    try:
        pois = get_poi_by_bbox(bbox_dict)

        if pois is not None: # Check if the API call returned data (even if empty list)
            pois_in_sa2_boundary = [] # Temp list for POIs truly within the SA2 boundary
            for poi in pois:
                try:
                    # Check if geometry exists and has x, y coordinates
                    if 'geometry' in poi and poi['geometry'] and 'x' in poi['geometry'] and 'y' in poi['geometry']:
                        # Create a shapely Point from the POI coordinates
                        poi_point = Point(poi['geometry']['x'], poi['geometry']['y'])

                        # Check if the POI point is within the actual SA2 polygon
                        if sa2_geometry.contains(poi_point):
                            # Add source SA2 information
                            if 'attributes' in poi:
                                poi['attributes']['sa2_code_source'] = sa2_code
                                poi['attributes']['sa2_name_source'] = sa2_name
                            else:
                                # Handle POIs that might lack an 'attributes' dictionary
                                poi['attributes'] = {'sa2_code_source': sa2_code, 'sa2_name_source': sa2_name}
                            pois_in_sa2_boundary.append(poi)
                    else:
                         print(f"- Warning: POI with missing/invalid geometry skipped: {poi.get('attributes', {}).get('objectid', 'N/A')}")

                except Exception as check_err:
                    print(f"- Error checking containment for POI {poi.get('attributes', {}).get('objectid', 'N/A')}: {check_err}")

            print(f"---> Found {len(pois)} POIs within BBox, kept {len(pois_in_sa2_boundary)} POIs within actual SA2 boundary for {sa2_name}")
            all_pois_in_selected_sa4.extend(pois_in_sa2_boundary) # Extend with the filtered list

        else:
            # This case handles if get_poi_by_bbox returned None due to an error
            print(f"---> Error fetching POIs for {sa2_name}. Skipping this SA2.")

    except Exception as e:
        # Catch any unexpected error during the API call for a specific SA2
        print(f"---> An unexpected error occurred fetching POIs for {sa2_name}: {e}")
        print("---> Skipping this SA2.")

print(f"\n\nFinished processing all SA2 regions in {selected_sa4_name}.")
print(f"Total POIs collected: {len(all_pois_in_selected_sa4)}")

Processing 19 SA2 regions in SA4: Sydney - Northern Beaches (122)

Fetching POIs for SA2: Balgowlah - Clontarf - Seaforth (122011418) using bbox...
---> Found 205 POIs within BBox, kept 87 POIs within actual SA2 boundary for Balgowlah - Clontarf - Seaforth

Fetching POIs for SA2: Manly - Fairlight (122011419) using bbox...
---> Found 196 POIs within BBox, kept 126 POIs within actual SA2 boundary for Manly - Fairlight

Fetching POIs for SA2: Avalon - Palm Beach (122021420) using bbox...
---> Found 186 POIs within BBox, kept 138 POIs within actual SA2 boundary for Avalon - Palm Beach

Fetching POIs for SA2: Bayview - Elanora Heights (122021421) using bbox...
---> Found 679 POIs within BBox, kept 221 POIs within actual SA2 boundary for Bayview - Elanora Heights

Fetching POIs for SA2: Newport - Bilgola (122021422) using bbox...
---> Found 278 POIs within BBox, kept 210 POIs within actual SA2 boundary for Newport - Bilgola

Fetching POIs for SA2: Mona Vale - Warriewood (North) (122021690) 

In [ ]:

# --- Task 2.iii  ---
# Next steps would involve processing 'all_pois_in_selected_sa4',
# converting it to a GeoDataFrame, cleaning,
# and ingesting it into the database.

# Extract attributes, handle potential missing 'attributes' key
poi_attributes_list = []
for poi in all_pois_in_selected_sa4:
    if 'attributes' in poi:
        poi_attributes_list.append(poi['attributes'])
    else:
        # Append a dictionary with at least source info if attributes are missing
        # This maintains alignment if geometry exists but attributes don't
        poi_attributes_list.append({'sa2_code_source': 'N/A', 'sa2_name_source': 'N/A'}) # Placeholder
        print(f"Warning: POI found with missing 'attributes'. POI data: {poi}")

poi_df = pd.DataFrame(poi_attributes_list)

poi_geometries = [Point(poi['geometry']['x'], poi['geometry']['y']) if poi.get('geometry') else None for poi in all_pois_in_selected_sa4]
poi_gdf_original = gpd.GeoDataFrame(poi_df, geometry=poi_geometries, crs="EPSG:7844")
poi_gdf = poi_gdf_original.copy()

print("\n=== POI DataFrame Head ===")
poi_gdf.head()


=== POI DataFrame Head ===


,objectid,topoid,poigroup,poitype,poiname,poilabel,poilabeltype,poialtlabel,poisourcefeatureoid,accesscontrol,...,lastupdate,msoid,centroidid,shapeuuid,changetype,processstate,urbanity,sa2_code_source,sa2_name_source,geometry
0,1094,500188493,1,Place Of Worship,None,METHODIST CHURCH,DERIVED,METHODIST,19,1,...,1285588392535,59793,None,21503b00-7b32-3f2d-89cf-0fa39ed828f0,I,None,U,122011418,Balgowlah - Clontarf - Seaforth,POINT (151.24686 -33.79747)
1,1096,500188560,3,Park,BANTRY RESERVE,BANTRY RESERVE,NAMED,None,61,1,...,1285588392535,84789,None,e5c51aec-bc1d-355e-bfb7-b4ce8219c13b,I,None,S,122011418,Balgowlah - Clontarf - Seaforth,POINT (151.24156 -33.78095)
2,1098,500188604,3,Park,BLIGH PARK,BLIGH PARK,NAMED,None,61,1,...,1285588392535,87887,None,f0866843-99be-386e-bb0d-344300537318,I,None,U,122011418,Balgowlah - Clontarf - Seaforth,POINT (151.24179 -33.78326)
3,2331,500277733,3,Park,MANLY WEST PARK,MANLY WEST PARK,NAMED,None,61,1,...,1414581788737,169809,None,a309b796-bccc-348a-8712-e4ccc417d972,M,None,U,122011418,Balgowlah - Clontarf - Seaforth,POINT (151.26916 -33.78868)
4,2418,500279477,3,Lookout,None,Lookout,GENERIC,None,56,1,...,1285588392535,83376,None,c3d77774-4544-3103-a4e4-aae004a3a96a,I,None,S,122011418,Balgowlah - Clontarf - Seaforth,POINT (151.26399 -33.81252)


In [ ]:
# Task 3 Prep: Remove unnecessary columns from poi_gdf for POI analysis

# Define the list of columns to remove
columns_to_remove = [
    'objectid', 'poilabeltype', 'poialtlabel', 'poisourcefeatureoid', 'accesscontrol',
    'suburbname', 'postcode', 'state', 'address', 'phonenumber', 'website', 'comment',
    'attributesource', 'attributedate', 'planimetricaccuracy', 'heightaccuracy',
    'featurecode', 'featuredescription', 'functiontype', 'functiondescription',
    'hierarchylevel', 'subhierarchylevel', 'source', 'capturedate', 'editdate',
    'editreason', 'lastupdate', 'msoid', 'centroidid', 'shapeuuid',
    'changetype', 'processstate', 'urbanity', 'startdate',	'enddate'
]

# Check which of these columns actually exist in the DataFrame to avoid errors
existing_columns_to_remove = [col for col in columns_to_remove if col in poi_gdf.columns]


poi_gdf_cleaned = poi_gdf.drop(columns=existing_columns_to_remove)
print(f"Removed {len(existing_columns_to_remove)} columns. Kept columns: {list(poi_gdf_cleaned.columns)}")


# For subsequent steps, use poi_gdf_cleaned
# Example: Display number of unique POI groups
if 'poigroup' in poi_gdf_cleaned.columns:
    print("\nUnique POI groups present:")
    print(poi_gdf_cleaned['poigroup'].value_counts().sort_index())
else:
    print("\n'poigroup' column not found in the cleaned DataFrame.")

print("\n=== Cleaned POI DataFrame Head ===")
poi_gdf_cleaned.head()


Removed 14 columns. Kept columns: ['topoid', 'poigroup', 'poitype', 'poiname', 'poilabel', 'sa2_code_source', 'sa2_name_source', 'geometry']

Unique POI groups present:
poigroup
1    388
2    100
3    788
4    232
5     14
6     50
7     69
8     89
9      2
Name: count, dtype: int64

=== Cleaned POI DataFrame Head ===


,topoid,poigroup,poitype,poiname,poilabel,sa2_code_source,sa2_name_source,geometry
0,500188493,1,Place Of Worship,None,METHODIST CHURCH,122011418,Balgowlah - Clontarf - Seaforth,POINT (151.24686 -33.79747)
1,500188560,3,Park,BANTRY RESERVE,BANTRY RESERVE,122011418,Balgowlah - Clontarf - Seaforth,POINT (151.24156 -33.78095)
2,500188604,3,Park,BLIGH PARK,BLIGH PARK,122011418,Balgowlah - Clontarf - Seaforth,POINT (151.24179 -33.78326)
3,500277733,3,Park,MANLY WEST PARK,MANLY WEST PARK,122011418,Balgowlah - Clontarf - Seaforth,POINT (151.26916 -33.78868)
4,500279477,3,Lookout,None,Lookout,122011418,Balgowlah - Clontarf - Seaforth,POINT (151.26399 -33.81252)


In [ ]:
from sqlalchemy import create_engine, text
import psycopg2
import psycopg2.extras
import json

credentials = "Credentials.json"

def pgconnect(credential_filepath, db_schema="public"):
    with open(credential_filepath) as f:
        db_conn_dict = json.load(f)
        host       = db_conn_dict['host']
        db_user    = db_conn_dict['user']
        db_pw      = db_conn_dict['password']
        default_db = db_conn_dict['user']
        port       = db_conn_dict['port']
        try:
            db = create_engine(f'postgresql+psycopg2://{db_user}:{db_pw}@{host}:{port}/{default_db}', echo=False)
            conn = db.connect()
            print('Connected successfully.')
        except Exception as e:
            print("Unable to connect to the database.")
            print(e)
            db, conn = None, None
        return db,conn

def query(conn, sqlcmd, args=None, df=True):
    result = pd.DataFrame() if df else None
    try:
        if df:
            result = pd.read_sql_query(sqlcmd, conn, params=args)
        else:
            result = conn.execute(text(sqlcmd), args).fetchall()
            result = result[0] if len(result) == 1 else result
    except Exception as e:
        print("Error encountered: ", e, sep='\n')
    return result

In [ ]:
db, conn = pgconnect(credentials)

Connected successfully.


In [ ]:
# Wrap the geometry column with WKTElement
sa2_gdf['geom'] = sa2_gdf['geometry'].apply(lambda x: create_wkt_element(geom=x,srid=7844))  # applying the function
sa2_gdf = sa2_gdf.drop(columns="geometry")
print(sa2_gdf.dtypes)

sa2_code     object
sa2_name     object
sa4_code     object
sa4_name     object
gcc_code     object
gcc_name     object
areasqkm    float64
geom         object
dtype: object


In [ ]:
table_creation_statements = [
        """
        CREATE TABLE IF NOT EXISTS sa2_regions (
            sa2_code TEXT PRIMARY KEY,
            sa2_name TEXT,
            sa4_code TEXT,
            sa4_name TEXT,
            gcc_code TEXT,
            gcc_name TEXT,
            areasqkm FLOAT,
            geometry GEOMETRY(MULTIPOLYGON, 7844)
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS businesses (
            industry_code TEXT NOT NULL,
            industry_name TEXT,
            sa2_code TEXT NOT NULL,
            sa2_name TEXT,
            "0_to_50k_businesses" INTEGER,
            "50k_to_200k_businesses" INTEGER,
            "200k_to_2m_businesses" INTEGER,
            "2m_to_5m_businesses" INTEGER,
            "5m_to_10m_businesses" INTEGER,
            "10m_or_more_businesses" INTEGER,
            total_businesses INTEGER,
            PRIMARY KEY (sa2_code, industry_code),
            FOREIGN KEY (sa2_code) REFERENCES sa2_regions(sa2_code)
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS transport_stops (
            stop_id TEXT PRIMARY KEY,
            stop_name TEXT,
            geometry GEOMETRY(POINT, 7844) NOT NULL
        );
        """,
        """
        DROP TABLE IF EXISTS school_catchments;
        CREATE TABLE IF NOT EXISTS school_catchments (
            school_code TEXT PRIMARY KEY,
            catchment_type TEXT,
            school_name TEXT,
            geometry GEOMETRY(MULTIPOLYGON, 7844) NOT NULL
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS population_sa2 (
            sa2_code TEXT PRIMARY KEY,
            sa2_name TEXT,
            "0-4_people" INTEGER,
            "5-9_people" INTEGER,
            "10-14_people" INTEGER,
            "15-19_people" INTEGER,
            "20-24_people" INTEGER,
            "25-29_people" INTEGER,
            "30-34_people" INTEGER,
            "35-39_people" INTEGER,
            "40-44_people" INTEGER,
            "45-49_people" INTEGER,
            "50-54_people" INTEGER,
            "55-59_people" INTEGER,
            "60-64_people" INTEGER,
            "65-69_people" INTEGER,
            "70-74_people" INTEGER,
            "75-79_people" INTEGER,
            "80-84_people" INTEGER,
            "85-and-over_people" INTEGER,
            total_people INTEGER,
            FOREIGN KEY (sa2_code) REFERENCES sa2_regions(sa2_code)
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS income_sa2 (
            sa2_code TEXT PRIMARY KEY,
            sa2_name TEXT,
            earners INTEGER,
            median_age INTEGER,
            median_income INTEGER,
            mean_income INTEGER,
            FOREIGN KEY (sa2_code) REFERENCES sa2_regions(sa2_code)
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS poi_sa2 (
            topoid TEXT PRIMARY KEY,
            poigroup TEXT,
            poitype TEXT,
            poiname TEXT,
            poilabel TEXT,
            sa2_code_source TEXT,
            sa2_name_source TEXT,
            geometry GEOMETRY(POINT, 7844) NOT NULL,
            FOREIGN KEY (sa2_code_source) REFERENCES sa2_regions(sa2_code)
        );
        """
    ]

In [ ]:
try:
    with conn.begin(): # Start a transaction
        for statement in table_creation_statements:
            conn.execute(text(statement))
            # Extract table name for logging (simple parsing for example)
            table_name = "Unknown"
            if "CREATE TABLE IF NOT EXISTS" in statement:
                table_name = statement.split("CREATE TABLE IF NOT EXISTS")[1].strip().split("(")[0].strip()
            print(f"Table '{table_name}' creation statement executed successfully.")
    print("\nAll tables created (or already existed) successfully.")
    conn.commit()
except Exception as e:
    print(f"An error occurred during table creation: {e}")

Table 'sa2_regions' creation statement executed successfully.
Table 'businesses' creation statement executed successfully.
Table 'transport_stops' creation statement executed successfully.
Table 'school_catchments' creation statement executed successfully.
Table 'population_sa2' creation statement executed successfully.
Table 'income_sa2' creation statement executed successfully.
Table 'poi_sa2' creation statement executed successfully.

All tables created (or already existed) successfully.


In [ ]:
#sa2_gdf.to_sql('sa2_regions', conn, if_exists='append', index=False, dtype={'geometry': Geometry('MULTIPOLYGON', 7844)})
stops_df.to_sql('transport_stops', conn, if_exists='append', index=False, dtype={'geometry': Geometry('POINT', 7844)})
#business.to_sql('businesses', conn, if_exists='append', index=False)
conn.commit()